# Protein Inspection with MolSysMT

This tutorial shows how to use the **molsysmt** addon to explore a protein structure inside MolSysViewer:
inspect atom/residue/chain counts, colour by a physicochemical property, and compute and overlay
hydrogen bonds — all from Python.

**Requirements:** `molsysmt`, `molsysviewer`, `molsysviewer-molsysmt`.

In [ ]:
import molsysmt as msm
import molsysviewer as msv
from molsysviewer_molsysmt import get_addon, lifecycle, on_enable
from molsysviewer_molsysmt.runtime import ensure_runtime

## Register the addon

Addons must be registered with the MolSysViewer addon registry before any view is created.
Registration is a one-time step per Python session.

In [ ]:
msv.addons.register(get_addon(), lifecycle=lifecycle)

## Load the molecular system

We load triosephosphate isomerase (TIM, PDB 1TCD) as a `molsysmt.MolSys` object.
The same system is passed to the viewer *and* to the addon runtime so that both share
an identical in-memory representation.

In [ ]:
ms = msm.convert("pdb_id:1tcd", to_form="molsysmt.MolSys")
msm.info(ms)

## Create the view and enable the addon

In [ ]:
view = msv.MolSysView()
view.load(ms)

# Attach the MolSysMT system to the addon runtime so panels can call msm.* on it.
runtime = ensure_runtime(view)
runtime.molecular_system = ms

on_enable(view)
view.show()

## Inspect system counts

The **System** panel widget can be driven programmatically via `handle_action`.
The call below mimics pressing the *Inspect* button in the panel UI.

In [ ]:
system_panel = view.addons.resolve_panel_widget("molsysmt", "system")
system_panel.handle_action(view, "inspect", {})

print(f"Atoms   : {runtime.n_atoms}")
print(f"Residues: {runtime.n_residues}")
print(f"Chains  : {runtime.n_chains}")
print(f"Frames  : {runtime.n_frames}")

## Colour by a physicochemical property

The **Color** panel maps a scalar per-residue property onto the viewer palette.
Here we colour by `charge` using the `RdBu` diverging colormap
(negative = blue, positive = red).

In [ ]:
color_panel = view.addons.resolve_panel_widget("molsysmt", "color")
color_panel.handle_action(
    view,
    "apply_color",
    {"property": "charge", "palette": "RdBu"},
)

## Compute and overlay hydrogen bonds

The **H-Bonds** panel calls `msm.hbonds.get_buch_hbonds()` and renders the results as
link shapes (dashed sticks) directly in the viewer.

In [ ]:
hbonds_panel = view.addons.resolve_panel_widget("molsysmt", "hbonds")
hbonds_panel.handle_action(view, "compute_hbonds", {})

print(f"H-bond shape tag: {runtime.hbonds_tag}")

## Export a static snapshot

Capture the current state — structure + colouring + H-bond overlays — as a
self-contained HTML file you can share or embed in documentation.

In [ ]:
view.export.html("1tcd_hbonds.html", title="TIM — H-bond network")

## Next steps

- Try other colour properties: `hydrophobicity`, `sasa`, `b_factor`, `secondary_structure`.
- Use the **Select** panel to highlight specific residue ranges.
- See `tutorial_trajectory_analysis.ipynb` to extend this workflow to a multi-frame trajectory.